# Inteligencia de Negocios

## Componente Práctico Semana 1

## Grupo # 3

### ALEJANDRA BEATRIZ TELLO GONZÁLEZ
### PABLO RAMIRO VALLEJO ZÚÑIGA


---
## Fase I – Configuración

### Dominio de negocio: **Ciberseguridad Bancaria**

El proyecto trabaja con datos del área de seguridad de la información en el sector financiero.  
Las tres fuentes de datos seleccionadas cubren distintas dimensiones de la operación de seguridad:

| # | Fuente | Tipo | Descripción |
|---|--------|------|-------------|
| 1 | `api_security_events` | PostgreSQL | Eventos de seguridad en canales bancarios digitales |
| 2 | `incidentes_seguridad.csv` | CSV | Registro de incidentes de red y ataques detectados |
| 3 | `vulnerabilidades_cve.json` | JSON | Resultado de escaneo de vulnerabilidades (CVE) |

### Infraestructura
- **Docker + PostgreSQL 17**: contenedor `ciber_12B` levantado con `docker-compose.yml`
- **Variables de entorno**: gestionadas con `python-dotenv` desde el archivo `.env`
- **IDE**: PyCharm Pro con environment virtual `Practica_G3`

---
## Fase II – Desarrollo

### Importación de dependencias

Se importan las librerías necesarias para:
- **pandas**: manipulación y análisis de DataFrames
- **sqlalchemy + psycopg2**: conexión ORM a PostgreSQL
- **json**: lectura del archivo de vulnerabilidades
- **dotenv**: carga de credenciales desde `.env` (buena práctica de seguridad)

In [ ]:
import pandas as pd
import json
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

load_dotenv()
print("✅ Dependencias cargadas correctamente")

### Manejo de variables de entorno

Las credenciales de la base de datos se leen del archivo `.env` para evitar exponer datos sensibles en el código fuente.  
Esta práctica es obligatoria en entornos bancarios donde los repositorios pueden ser auditados.

In [ ]:
DB_USER     = os.getenv("POSTGRES_USER")
DB_PASSWORD = os.getenv("POSTGRES_PASSWORD")
DB_NAME     = os.getenv("POSTGRES_DB")
DB_HOST     = os.getenv("POSTGRES_HOST")
DB_PORT     = os.getenv("POSTGRES_PORT", "5432")

print(f"🔌 Conectando a: {DB_HOST}:{DB_PORT}/{DB_NAME} como usuario '{DB_USER}'")

### Conexión y carga de la base de datos PostgreSQL

Se crea el engine de SQLAlchemy apuntando al contenedor Docker.
Luego se ejecuta el script SQL de seed para poblar la tabla `api_security_events`,
que contiene eventos de seguridad registrados en los canales bancarios digitales.

In [ ]:
engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

# Cargar el script SQL de seed (crear tabla + insertar 150 registros)
with open("data/seed_api_security.sql", "r") as f:
    sql_seed = f.read()

with engine.connect() as conn:
    for statement in sql_seed.split(";"):
        stmt = statement.strip()
        if stmt and not stmt.startswith("--"):
            conn.execute(text(stmt))
    conn.commit()

print("✅ Tabla 'api_security_events' creada y datos cargados")

---
## Generación de los tres DataFrames principales

En esta sección se extraen los datos de cada una de las tres fuentes definidas en la Fase I.
Cada DataFrame representa una vista diferente del ecosistema de ciberseguridad:

- `df_postgres` → eventos de seguridad en APIs bancarias (fuente: PostgreSQL)
- `df_csv` → incidentes de red detectados por el SOC (fuente: CSV)
- `df_json` → vulnerabilidades CVE identificadas en escaneos (fuente: JSON)

In [ ]:
# ── DataFrame 1: desde PostgreSQL ──────────────────────────────────────────
df_postgres = pd.read_sql("SELECT * FROM api_security_events", engine)

# ── DataFrame 2: desde CSV ─────────────────────────────────────────────────
df_csv = pd.read_csv("data/incidentes_seguridad.csv")

# ── DataFrame 3: desde JSON ────────────────────────────────────────────────
with open("data/vulnerabilidades_cve.json", "r", encoding="utf-8") as f:
    vulnerabilidades = json.load(f)
df_json = pd.DataFrame(vulnerabilidades)

print(f"✅ df_postgres : {df_postgres.shape[0]} filas × {df_postgres.shape[1]} columnas")
print(f"✅ df_csv      : {df_csv.shape[0]} filas × {df_csv.shape[1]} columnas")
print(f"✅ df_json     : {df_json.shape[0]} filas × {df_json.shape[1]} columnas")

---
## Script de Análisis 1 – API Security Events (PostgreSQL)

Este DataFrame contiene **150 eventos** de seguridad registrados en los canales digitales del banco  
(Cash Management, Mobile Banking, Internet Banking, etc.).  
Es la fuente más crítica porque refleja el comportamiento real de las APIs expuestas al cliente.

In [ ]:
# ── i. Primeras 5 filas ────────────────────────────────────────────────────
# Se visualiza la estructura inicial del DataFrame para validar que la extracción
# desde PostgreSQL fue exitosa y los tipos de datos son correctos.
print("── Primeras 5 filas ──")
df_postgres.head()

In [ ]:
# ── ii. Columnas con valores nulos ─────────────────────────────────────────
# En un entorno bancario, los nulos pueden indicar datos de transacciones incompletas
# o eventos donde el agente HTTP no fue capturado (ej: tráfico automatizado).
nulos_pg = df_postgres.isnull().any()
print("Columnas con valores nulos:")
print(nulos_pg[nulos_pg == True])

In [ ]:
# ── iii. Conteo de valores faltantes por columna ───────────────────────────
# Permite priorizar la limpieza de datos: columnas con muchos nulos
# requieren estrategias de imputación antes de la fase de transformación ETL.
print("Valores faltantes por columna:")
print(df_postgres.isnull().sum()[df_postgres.isnull().sum() > 0])

In [ ]:
# ── iv. Estadísticas de variable numérica: response_time_ms ───────────────
# El tiempo de respuesta es un KPI de seguridad: tiempos elevados pueden indicar
# ataques de DoS, consultas inyectadas o sobrecarga del API Gateway.
stats_rt = df_postgres["response_time_ms"].agg(["mean", "max", "min"])
print("Estadísticas de tiempo de respuesta (ms):")
print(f"  Promedio : {stats_rt['mean']:.2f} ms")
print(f"  Máximo   : {stats_rt['max']} ms")
print(f"  Mínimo   : {stats_rt['min']} ms")

In [ ]:
# ── v. Valor máximo y mínimo de risk_score agrupado por channel y event_type
# Identifica qué combinación de canal + tipo de evento presenta mayor y menor riesgo.
# Información clave para priorizar controles de seguridad por canal.
riesgo_por_grupo = (
    df_postgres
    .groupby(["channel", "event_type"])["risk_score"]
    .agg(["max", "min"])
    .reset_index()
    .rename(columns={"max": "risk_max", "min": "risk_min"})
    .sort_values("risk_max", ascending=False)
)
print("Máximo y mínimo de risk_score por canal y tipo de evento:")
print(riesgo_por_grupo.to_string(index=False))

---
## Script de Análisis 2 – Incidentes de Seguridad de Red (CSV)

Este DataFrame contiene **300 incidentes** registrados por el sistema de detección de intrusos (IDS).  
Incluye metadatos de red como protocolo, bytes transferidos, tipo de ataque y severidad.  
Es útil para analizar patrones de ataque y evaluar la efectividad de los controles perimetrales.

In [ ]:
# ── i. Primeras 5 filas ────────────────────────────────────────────────────
# Se verifica que el CSV fue leído correctamente con todos sus campos.
# Importante confirmar que 'timestamp' se cargó como string (se tipará en ETL).
print("── Primeras 5 filas ──")
df_csv.head()

In [ ]:
# ── ii. Columnas con valores nulos ─────────────────────────────────────────
# La columna 'analyst_assigned' puede tener nulos cuando el incidente
# fue manejado automáticamente por el SIEM sin intervención humana.
nulos_csv = df_csv.isnull().any()
print("Columnas con valores nulos:")
print(nulos_csv[nulos_csv == True])

In [ ]:
# ── iii. Conteo de valores faltantes por columna ───────────────────────────
# Un analista no asignado (~10% esperado) no es un error: indica automatización.
# En la fase Transform se decidirá si imputar con 'Sistema Automático' o dejar nulo.
print("Valores faltantes por columna:")
total_nulos = df_csv.isnull().sum()
print(total_nulos[total_nulos > 0])

In [ ]:
# ── iv. Estadísticas de variable numérica: bytes_transferred ──────────────
# El volumen de datos transferidos es un indicador de exfiltración de información.
# Valores máximos muy elevados pueden corresponder a ataques de exfiltración masiva.
stats_bytes = df_csv["bytes_transferred"].agg(["mean", "max", "min"])
print("Estadísticas de bytes transferidos:")
print(f"  Promedio : {stats_bytes['mean']:,.0f} bytes ({stats_bytes['mean']/1024:.1f} KB)")
print(f"  Máximo   : {stats_bytes['max']:,} bytes ({stats_bytes['max']/1048576:.2f} MB)")
print(f"  Mínimo   : {stats_bytes['min']:,} bytes")

In [ ]:
# ── v. Máximo y mínimo de duration_seconds agrupado por attack_type y severity
# Ataques de larga duración con severidad Critical son los más peligrosos.
# Esta vista ayuda al SOC a calibrar los umbrales de alerta por tipo de ataque.
duracion_grupo = (
    df_csv
    .groupby(["attack_type", "severity"])["duration_seconds"]
    .agg(["max", "min"])
    .reset_index()
    .rename(columns={"max": "duracion_max_seg", "min": "duracion_min_seg"})
    .sort_values("duracion_max_seg", ascending=False)
)
print("Duración máxima y mínima (segundos) por tipo de ataque y severidad:")
print(duracion_grupo.to_string(index=False))

---
## Script de Análisis 3 – Vulnerabilidades CVE (JSON)

Este DataFrame contiene **80 vulnerabilidades** identificadas en el último escaneo de infraestructura.  
Cada registro incluye el CVE ID, score CVSS, sistema afectado y estado del parche.  
En el sector bancario, el patch management tiene plazos regulatorios definidos por la SB (Superintendencia de Bancos).

In [ ]:
# ── i. Primeras 5 filas ────────────────────────────────────────────────────
# Se verifica la estructura del JSON normalizado.
# El campo 'patch_applied' puede contener None (nulo) para sistemas sin diagnóstico.
print("── Primeras 5 filas ──")
df_json.head()

In [ ]:
# ── ii. Columnas con valores nulos ─────────────────────────────────────────
# Los nulos en 'patch_applied' indican vulnerabilidades donde el estado
# del parche no ha sido verificado aún — requieren seguimiento urgente.
nulos_json = df_json.isnull().any()
print("Columnas con valores nulos:")
print(nulos_json[nulos_json == True])

In [ ]:
# ── iii. Conteo de valores faltantes por columna ───────────────────────────
# Se cuantifica cuántas vulnerabilidades tienen estado de parche desconocido.
# En auditorías PCI-DSS, esto cuenta como hallazgo de control.
print("Valores faltantes por columna:")
total_nulos_json = df_json.isnull().sum()
print(total_nulos_json[total_nulos_json > 0])

In [ ]:
# ── iv. Estadísticas de variable numérica: cvss_score ─────────────────────
# El CVSS (Common Vulnerability Scoring System) va de 0 a 10.
# Un promedio > 7 indica una infraestructura con exposición alta.
# El máximo y mínimo permiten identificar los extremos del portafolio de riesgo.
stats_cvss = df_json["cvss_score"].agg(["mean", "max", "min"])
print("Estadísticas del score CVSS:")
print(f"  Promedio : {stats_cvss['mean']:.2f} / 10")
print(f"  Máximo   : {stats_cvss['max']} / 10")
print(f"  Mínimo   : {stats_cvss['min']} / 10")

In [ ]:
# ── v. Máximo y mínimo de affected_hosts agrupado por system y cvss_severity
# Permite priorizar el parcheo: vulnerabilidades con severidad Critical
# que afectan más hosts deben resolverse primero (ISO 27001 / Ley de Defensa del Consumidor).
hosts_grupo = (
    df_json
    .groupby(["system", "cvss_severity"])["affected_hosts"]
    .agg(["max", "min"])
    .reset_index()
    .rename(columns={"max": "hosts_max", "min": "hosts_min"})
    .sort_values("hosts_max", ascending=False)
)
print("Hosts afectados (máx/mín) por sistema y severidad CVSS:")
print(hosts_grupo.to_string(index=False))

---
# Aplicación Profesional de la Práctica
## Alejandra Beatriz Tello González

Me desempeño como Líder de Seguridad de la Información en el sector bancario ecuatoriano, con enfoque específico en pruebas de penetración a APIs y cumplimiento normativo. Esta práctica me resultó particularmente relevante porque conecta directamente con los desafíos técnicos que enfrento en mi trabajo diario.

En el entorno bancario se generan y gestionan datos de naturaleza muy diversa: registros de transacciones financieras, logs de autenticación en canales digitales (banca móvil, Cash Management, Open Banking), eventos de seguridad capturados por el SIEM, resultados de escaneos de vulnerabilidades, alertas del WAF y del IDS/IPS, y métricas de disponibilidad de servicios críticos. Todos estos flujos producen volúmenes masivos de información que históricamente han sido analizados de forma aislada, sin una visión integrada que permita correlacionar eventos entre capas.

La combinación de PostgreSQL, Docker y Python que trabajamos en esta práctica ofrece una respuesta concreta a esa fragmentación. PostgreSQL permite centralizar los eventos de seguridad en una estructura relacional auditada y persistente, ideal para soportar consultas regulatorias (SB, PCI-DSS). Docker garantiza que el entorno de análisis sea reproducible en cualquier nodo del equipo de seguridad sin conflictos de dependencias, lo que es crítico cuando se trabaja con analistas de distintos perfiles. Python, con pandas y SQLAlchemy, proporciona la flexibilidad para transformar datos de fuentes heterogéneas —logs en JSON, reportes en CSV, consultas directas a base de datos— en DataFrames unificados y procesables.

Implementar un proceso ETL en Banco General Rumiñahui, donde actualmente realizo trabajo de seguridad, aportaría beneficios concretos: consolidar en un solo repositorio los eventos del WAF, los resultados del pentesting a APIs y los logs de Cash Management eliminaría el análisis manual en silos que hoy consume horas de trabajo. La fase de transformación permitiría estandarizar formatos de timestamp, clasificar eventos por severidad CVSS y cruzar incidentes de red con el inventario de vulnerabilidades activas.

Las decisiones que podrían tomarse con esta información son de alto impacto: priorizar el parcheo de sistemas con vulnerabilidades Critical explotadas actualmente, detectar patrones de ataques de fuerza bruta correlacionados con horarios de menor monitoreo, identificar endpoints API con tiempos de respuesta anómalos que podrían indicar inyección en curso, y justificar con datos cuantificables la inversión en controles adicionales ante la Gerencia de Riesgos. En un sector donde una brecha de seguridad implica sanciones regulatorias, pérdida de confianza del cliente y potencial daño reputacional, la capacidad de analizar datos de seguridad de forma automatizada e integrada no es una ventaja competitiva: es una necesidad operativa.